# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 3, 'num_retries': 0}


## 3. Schema enforcement — try to write a wrong schema

In [4]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
version_before_bad = DeltaTable(table_path).version()
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    bad_write_blocked = False
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    bad_write_blocked = True
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")
# A blocked write must not leave a commit behind.
version_after_bad = DeltaTable(table_path).version()
print(f"table version before/after bad write: {version_before_bad} → {version_after_bad}")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type
table version before/after bad write: 0 → 0


## 4. Schema evolution (opt-in)

In [5]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [6]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## 6. Evidence — the transaction log on disk

Every commit is one JSON file; each line is one action (`commitInfo`,
`protocol`, `metaData`, `add`). The second commit is the schema-merge append.

In [7]:
import json as _json
from pathlib import Path as _P

_log_dir = _P(table_path) / "_delta_log"
for _f in sorted(_log_dir.glob("*.json")):
    print(f"{_f.name}  ({_f.stat().st_size} B)")
    for _line in _f.read_text(encoding="utf-8").splitlines():
        _a = _json.loads(_line)
        _k = next(iter(_a))
        if _k == "add":
            print(f"   add       path={_a['add']['path'][:48]}…  size={_a['add']['size']}  stats={_a['add']['stats'][:70]}…")
        elif _k == "metaData":
            _fields = [f["name"] for f in _json.loads(_a["metaData"]["schemaString"])["fields"]]
            print(f"   metaData  schema={_fields}")
        elif _k == "commitInfo":
            print(f"   commitInfo operation={_a['commitInfo'].get('operation')}  mode={_a['commitInfo'].get('operationParameters', {}).get('mode')}")
        else:
            print(f"   {_k:9s} {_json.dumps(_a[_k])[:90]}")
print("\nSchema after evolution:", DeltaTable(table_path).schema().to_arrow())

00000000000000000000.json  (1409 B)
   commitInfo operation=WRITE  mode=Overwrite
   protocol  {"minReaderVersion": 1, "minWriterVersion": 2}
   metaData  schema=['id', 'name', 'age', 'city']
   add       path=part-00000-e405a7d5-70d2-4c39-b269-af7764bd8416-…  size=1384  stats={"numRecords":3,"minValues":{"id":1,"age":25,"name":"alice","city":"Da…
00000000000000000001.json  (1465 B)
   commitInfo operation=WRITE  mode=Append
   metaData  schema=['id', 'name', 'age', 'city', 'tier']
   add       path=part-00000-d1a17e6e-4545-4bdb-82b3-4c7a5e25bbcb-…  size=1548  stats={"numRecords":1,"minValues":{"city":"Hue","tier":"premium","age":28,"i…

Schema after evolution: arro3.core.Schema
------------
id: Int64
name: Utf8
age: Int64
city: Utf8
tier: Utf8



## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The schema-enforcement flag is now taken from the bad-write cell itself
(exception raised AND no new table version), not hardcoded.

In [8]:
from pathlib import Path as _Path  # noqa: E402

_log = sorted(_Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": bad_write_blocked and version_after_bad == version_before_bad,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.


## Kết quả & giải thích (Đỗ Quang Vinh)

| Kiểm tra | Kết quả đo được |
|---|---|
| Commit JSON trong `_delta_log/` | 2 file: `00000…0.json` (WRITE/Overwrite, có `protocol` + `metaData` + `add`) và `00000…1.json` (WRITE/Append, `metaData` mới có thêm `tier`) |
| Ghi `age='thirty'` | Bị chặn: `Cast error: Cannot cast string 'thirty' to value of Int64 type`; version bảng **0 → 0**, tức không có commit nào được tạo |
| `schema_mode="merge"` | Schema mới `id, name, age, city, tier`; 3 dòng cũ có `tier = null` |
| DuckDB đếm tier | `[('premium', 1), (None, 3)]` → 2 nhóm |

**Enforcement khác evolution ở điểm nào?** Enforcement là mặc định: mỗi lần ghi, writer so schema của dữ liệu mới với `metaData.schemaString` trong log; kiểu không khớp (string vào cột Int64) thì cả transaction bị huỷ và không có file JSON mới. Evolution là thay đổi *chính schema của bảng*: cùng lần ghi đó tạo thêm một action `metaData` mới trong commit.

**Vì sao thêm cột phải opt-in?** Một cột lạ trong dữ liệu đến thường là lỗi upstream (đổi tên trường, gõ sai) chứ không phải ý định. Nếu tự động thêm, schema sẽ âm thầm phình ra và reader downstream nhận cột mà không ai duyệt. `schema_mode="merge"` buộc người ghi tuyên bố rõ "tôi muốn đổi schema".

**Transaction log chứng minh gì về một lần ghi?** Mỗi version là một file JSON được ghi nguyên tử. Nó cho biết *ai/khi nào/thao tác gì* (`commitInfo`: operation, mode), *schema hiệu lực* (`metaData`), và *chính xác file Parquet nào thuộc version đó* (`add` kèm size và stats min/max/numRecords). Reader chỉ tin các file được `add` trong log, nên lần ghi bị chặn không để lại dấu vết.

> Ghi chú: cờ enforcement ở cuối notebook gốc bị hardcode `True`. Mình đã thay bằng biến `bad_write_blocked`, lấy từ cell ghi sai kiểu, kèm kiểm tra version không đổi.